In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

data = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")

data.to_csv("submission.csv", index=False)

# Milestone 1

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
os.listdir("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems")

In [ ]:
# ================= CONFIGURATION =================

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

GENRES = [
    "blues",
    "classical",
    "country",
    "disco",
    "hiphop",
    "jazz",
    "metal",
    "pop",
    "reggae",
    "rock"
]

STEMS = {
    "drums.wav": "drums",
    "vocals.wav": "vocals",
    "bass.wav": "bass",
    "other.wav": "other"   
}

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']

GENRE_TO_TEST = "rock"

SONG_INDEX = 0   # First rock song (used in Q10–Q12)


**Q1)Complete the function `build_dataset` in question notebook and answer following questions (Q1 to Q3).
Hint: 1kb = 1024 bytes**

**What is the value of total number of corrupted sounds ( less than 4kb) + (total number of sounds < 5.0491MB)**

* 0
* 1256
* 1257
* 1259
* 3816

**Q2)What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB ?**

* 1440
* 3816
* 1257
* 184
* 1072
  
**Q3)What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?**

* 83
* 17
* 66
* 56

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):

    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    corrupted_count = 0
    less_50491_count = 0
    greater_50493_count = 0

    SIZE_4KB = 4 * 1024
    SIZE_50491 = 5.0491 * 1024 * 1024
    SIZE_50493 = 5.0493 * 1024 * 1024

    for genre in GENRES:

        genre_path = os.path.join(root_dir, genre)
        songs = sorted(os.listdir(genre_path))

        valid_songs = []

        for song in songs:

            song_path = os.path.join(genre_path, song)

            if not os.path.isdir(song_path):
                continue

            stem_files = os.listdir(song_path)

            # Must contain all 4 stems
            if set(stem_files) != set(STEMS.keys()):
                continue

            complete = True

            for stem_file in STEMS.keys():

                file_path = os.path.join(song_path, stem_file)
                file_size = os.path.getsize(file_path)

                # Corrupted
                if file_size < SIZE_4KB:
                    corrupted_count += 1

                # < 5.0491MB
                if file_size < SIZE_50491:
                    less_50491_count += 1

                # > 5.0493MB
                if file_size > SIZE_50493:
                    greater_50493_count += 1

            valid_songs.append(song)

        # Stratified split
        rng.shuffle(valid_songs)
        val_size = int(len(valid_songs) * val_split)

        val_songs = valid_songs[:val_size]
        train_songs = valid_songs[val_size:]

        # Add to dictionaries
        for song in train_songs:
            song_path = os.path.join(genre_path, song)
            for stem_file, stem_key in STEMS.items():
                train_dataset[genre][stem_key].append(
                    os.path.join(song_path, stem_file)
                )

        for song in val_songs:
            song_path = os.path.join(genre_path, song)
            for stem_file, stem_key in STEMS.items():
                val_dataset[genre][stem_key].append(
                    os.path.join(song_path, stem_file)
                )

    print("Corrupted sounds:", corrupted_count)
    print("Sounds < 5.0491MB:", less_50491_count)
    print("Sounds > 5.0493MB:", greater_50493_count)

    print("Absolute difference Q2:",
          abs(greater_50493_count - less_50491_count))

    print("Train reggae drums:",
          len(train_dataset["reggae"]["drums"]))

    print("Val country vocals:",
          len(val_dataset["country"]["vocals"]))

    print("Absolute difference Q3:",
          abs(len(train_dataset["reggae"]["drums"]) -
              len(val_dataset["country"]["vocals"])))

    return train_dataset, val_dataset


In [ ]:
tr, val = build_dataset(DATA_ROOT)


**Q4)Complete the function `find_long_silences` in question notebook and answer following questions (Q4 to Q9).**

**Total number of sound files having silence greater than equal to 5 secs.**

* 876
* 570
* 967
* 658
* 676
  
**Q5)Total number of sound tracks in Vocals where silence >= 5 secs**

* 304
* 194
* 174
* 314
* 276
 
**Q6)What's the average Silence Length in Vocals (in secs).**

* 11.50
* 5.00
* 12.79
* 10.56
* 13.11
  
**Q7)Total number of drums sound tracks in jazz where silence >= 5 secs**

* 9
* 21
* 19
* 14
* 17
  
**Q8)Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle.**

* 19
* 15
* 14
* 11
* 13
  
**Q9)Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10.**

* 7
* 6
* 5
* 4
* 3

In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):

    records = []
    total_files = 0

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem_name]:

                total_files += 1

                y, sr = librosa.load(file_path, sr=sr)

                total_duration = librosa.get_duration(y=y, sr=sr)

                intervals = librosa.effects.split(
                    y,
                    top_db=top_db,
                    frame_length=N_FFT,
                    hop_length=HOP_LENGTH
                )

                silence_segments = []

                # Fully silent
                if len(intervals) == 0:
                    silence_segments.append(total_duration)

                else:
                    # Start silence
                    start_silence = intervals[0][0] / sr
                    if start_silence > 0:
                        silence_segments.append(start_silence)

                    # Middle silence
                    for i in range(1, len(intervals)):
                        gap = (intervals[i][0] - intervals[i-1][1]) / sr
                        if gap > 0:
                            silence_segments.append(gap)

                    # End silence
                    end_silence = (len(y) - intervals[-1][1]) / sr
                    if end_silence > 0:
                        silence_segments.append(end_silence)

                max_silence = max(silence_segments) if silence_segments else 0

                silence_type = []

                if len(intervals) == 0:
                    silence_type.append("fully")
                else:
                    if intervals[0][0] > 0:
                        silence_type.append("start")
                    if intervals[-1][1] < len(y):
                        silence_type.append("end")
                    if len(intervals) > 1:
                        silence_type.append("middle")

                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(silence_type),
                        "File_Path": file_path
                    })

    print("Total files checked:", total_files)

    return pd.DataFrame(records)


In [ ]:
df_silence = find_long_silences(tr, threshold_sec=5.0, top_db=20)


In [ ]:
print(len(df_silence))
print(len(df_silence[df_silence["Stem"] == "vocals"]))
print(round(df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"].mean(), 2))

print(len(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums")
]))

print(len(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "middle")
]))

print(len(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
]))


In [ ]:
print(len(df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"].str.contains("middle"))
]))


**Q10)Select the first song from the ‘rock’ genre, combine all stems to prepare a sample, perform the tasks outlined in the notebook, and answer questions 10–12 based on the results.**

**What is the length of the mix sample?**

* 110150
* 110250
* 100250
* 120250
* 110251
  
**Q11)What is the value of RMS Amplitude of mix sample?**

* 0.12
* 0.17
* 0.22
* 0.19
* 0.11
  
**Q12)What is the value of max value of peak  normalized sample ?**

* 0.85
* 0.89
* 0.71
* 0.58
* 0.78

In [ ]:
stems_audio = []

try:
    for key in STEM_KEYS:
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]

        y, sr = librosa.load(
            file_path,
            sr=SR,
            duration=DURATION   # 5 seconds only
        )

        stems_audio.append(y)

    print("Audio loaded successfully.")

except NameError:
    print("ERROR: 'tr' dictionary not found.")
except IndexError:
    print("ERROR: Song index out of range.")
except Exception as e:
    print(f"ERROR: {e}")


In [ ]:
rock_path = os.path.join(DATA_ROOT, "rock")
songs = sorted(os.listdir(rock_path))

first_song = songs[0]
song_path = os.path.join(rock_path, first_song)

stems_audio = []

for stem_file in STEMS.keys():
    file_path = os.path.join(song_path, stem_file)

    y, sr = librosa.load(
        file_path,
        sr=SR,
        duration=DURATION
    )

    stems_audio.append(y)


In [ ]:
stems_stack = np.vstack(stems_audio)
mix_raw = np.sum(stems_stack, axis=0)

print("Length:", len(mix_raw))

rms_val = np.sqrt(np.mean(mix_raw ** 2))
print("RMS:", round(rms_val, 2))

max_val = np.max(np.abs(mix_raw))
print("Max before normalization:", round(max_val, 2))


In [ ]:
stems_stack = np.vstack(stems_audio)
print("Shape:", stems_stack.shape)


In [ ]:
rms_val = np.sqrt(np.mean(mix_raw ** 2))
print("RMS:", round(rms_val, 2))


In [ ]:
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

print("Max of normalized:", round(np.max(np.abs(mix_norm)), 2))
